In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q onnx onnxruntime scipy

Mounted at /content/drive


In [ ]:
!pip install -q onnx onnxscript onnxruntime

In [ ]:
import os

matches = []
for root, dirs, _ in os.walk("/content/drive/MyDrive"):
    if "1_Unloaded_Condition" in dirs or "2_Loaded_Condition" in dirs:
        matches.append((root, sorted(dirs)))

for parent, subdirs in matches:
    print("Parent folder:", parent)
    print("  Subfolders:", subdirs)

Parent folder: /content/drive/MyDrive/University of Ottawa Electric Motor Dataset – Vibr/2_CSV_Data_Files
  Subfolders: ['1_Unloaded_Condition', '2_Loaded_Condition']
Parent folder: /content/drive/MyDrive/University of Ottawa Electric Motor Dataset – Vibr/3_MatLab_Data_Files
  Subfolders: ['1_Unloaded_Condition', '2_Loaded_Condition']


In [ ]:
import os, glob
import scipy.io as sio
import numpy as np

# Path to your uploaded .mat files in Google Drive
DATASET_DIR = "/content/drive/MyDrive/University of Ottawa Electric Motor Dataset – Vibr/3_MatLab_Data_Files"

LOCAL_ONNX_PATH = "/content/motor_fault_classifier.onnx"
DRIVE_ONNX_PATH = "/content/drive/MyDrive/motor_fault_classifier.onnx"

mat_files = sorted(glob.glob(os.path.join(DATASET_DIR, "**", "*.mat"), recursive=True))
print(f"Found {len(mat_files)} .mat files")

if mat_files:
    sample_mat = sio.loadmat(mat_files[0])
    arrays = [v for v in sample_mat.values() if isinstance(v, np.ndarray) and v.ndim == 2]
    arr = max(arrays, key=lambda a: a.size)
    print(f"Sample file: {os.path.basename(mat_files[0])} | Array shape: {arr.shape}")

Found 128 .mat files
Sample file: B_R_1_0.mat | Array shape: (420000, 5)


In [ ]:
import re
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix

CLASSES = ["H-H", "R-U", "R-M", "B-R", "K-A", "S-W", "V-U", "F-B"]
LABEL_TO_IDX = {label: idx for idx, label in enumerate(CLASSES)}

SAMPLE_RATE = 42000
WINDOW_SAMPLES = 21000  # 0.5 s @ 42,000 Hz
HOP_SAMPLES = 10500     # 0.25 s step (50% overlap)
NUM_CHANNELS = 4        # 0: Accel 1, 1: Accel 2, 2: Accel 3, 3: Microphone

# Dataset order: 0=Accel1, 1=Mic, 2=Accel2, 3=Accel3, 4=Temp -> Contract order: [0, 2, 3, 1]
RAW_TO_CONTRACT_CHANNELS = [0, 2, 3, 1]

def parse_fault_label(filename: str):
    stem = os.path.splitext(os.path.basename(filename))[0].upper()
    # Matches patterns like B_R_1_0 or B-R-1-0
    match = re.match(r"^([A-Z])[_-]([A-Z])[_-]\d+[_-]\d+", stem)
    if match:
        code = f"{match.group(1)}-{match.group(2)}"
        if code in LABEL_TO_IDX:
            return code
    # Fallback: normalize underscores to hyphens and search for class codes
    norm = stem.replace("_", "-")
    for code in CLASSES:
        if re.search(rf"(^|[^A-Z0-9]){re.escape(code)}([^A-Z0-9]|$)", norm):
            return code
    return None

def load_recording_4ch(filepath: str):
    mat = sio.loadmat(filepath)
    arrays = [
        v for v in mat.values()
        if isinstance(v, np.ndarray) and v.ndim == 2 and np.issubdtype(v.dtype, np.number)
    ]
    if not arrays:
        return None
    arr = max(arrays, key=lambda a: a.size).astype(np.float32)
    if arr.shape[0] != 5 and arr.shape[1] == 5:
        arr = arr.T
    if arr.shape[0] < 5:
        return None
    return arr[RAW_TO_CONTRACT_CHANNELS, :]

def split_files_by_recording(dataset_dir: str, train_ratio=0.70, val_ratio=0.15, seed=42):
    all_files = []
    for root, _, files in os.walk(dataset_dir):
        for f in sorted(files):
            if f.lower().endswith(".mat") and not f.startswith("."):
                label = parse_fault_label(f)
                if label is not None:
                    all_files.append((os.path.join(root, f), LABEL_TO_IDX[label]))

    rng = np.random.default_rng(seed)
    train_files, val_files, test_files = [], [], []
    for cls_idx in range(len(CLASSES)):
        cls_items = [item for item in all_files if item[1] == cls_idx]
        if not cls_items:
            continue
        perm = rng.permutation(len(cls_items))
        n = len(cls_items)
        n_val = max(1, int(round(n * val_ratio)))
        n_test = max(1, int(round(n * (1.0 - train_ratio - val_ratio))))
        if n_val + n_test >= n:
            n_test = 1
            n_val = 1 if n >= 3 else 0
        n_train = n - n_val - n_test

        train_files.extend(cls_items[i] for i in perm[:n_train])
        val_files.extend(cls_items[i] for i in perm[n_train : n_train + n_val])
        test_files.extend(cls_items[i] for i in perm[n_train + n_val :])

    return train_files, val_files, test_files

def build_window_dataset(file_label_pairs):
    X_list, y_list = [], []
    for path, label_idx in file_label_pairs:
        sig_4xN = load_recording_4ch(path)
        if sig_4xN is None or sig_4xN.shape[1] < WINDOW_SAMPLES:
            continue
        n_win = 1 + (sig_4xN.shape[1] - WINDOW_SAMPLES) // HOP_SAMPLES
        for w in range(n_win):
            start = w * HOP_SAMPLES
            X_list.append(sig_4xN[:, start : start + WINDOW_SAMPLES])
            y_list.append(label_idx)

    if not X_list:
        return (
            np.empty((0, NUM_CHANNELS, WINDOW_SAMPLES), dtype=np.float32),
            np.empty((0,), dtype=np.int64),
        )

    return np.stack(X_list, axis=0).astype(np.float32), np.array(y_list, dtype=np.int64)
train_files, val_files, test_files = split_files_by_recording(DATASET_DIR)
print(f"Recordings -> Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}")

X_train, y_train = build_window_dataset(train_files)
X_val, y_val = build_window_dataset(val_files)
X_test, y_test = build_window_dataset(test_files)
print(f"Windows -> Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Recordings -> Train: 96, Val: 16, Test: 16
Windows -> Train: (3744, 4, 21000), Val: (624, 4, 21000), Test: (624, 4, 21000)


In [ ]:
#code for model version 1 (base model - Single-scale 1D-CNN)

class MotorFault1DCNN(nn.Module):
    def __init__(self, channel_scales: np.ndarray, num_classes: int = len(CLASSES)):
        super().__init__()
        scales = np.asarray(channel_scales, dtype=np.float32).reshape(1, NUM_CHANNELS, 1)
        inv_scale = 1.0 / np.maximum(scales, 1e-8)
        self.register_buffer("inv_scale", torch.tensor(inv_scale, dtype=torch.float32))

        self.conv_block = nn.Sequential(
            nn.Conv1d(NUM_CHANNELS, 32, kernel_size=64, stride=4, padding=32, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.AvgPool1d(kernel_size=4, stride=4),

            nn.Conv1d(32, 64, kernel_size=16, stride=2, padding=8, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.AvgPool1d(kernel_size=4, stride=4),

            nn.Conv1d(64, 128, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool1d(1),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(64, num_classes),
        )
        self.softmax = nn.Softmax(dim=-1)

    def encode_logits(self, signal: torch.Tensor) -> torch.Tensor:
        # Remove per-window, per-channel DC offset; keep amplitude scale intact
        centered = signal - signal.mean(dim=-1, keepdim=True)
        scaled = centered * self.inv_scale
        return self.head(self.conv_block(scaled))

    def forward(self, signal: torch.Tensor) -> torch.Tensor:
        return self.softmax(self.encode_logits(signal))

class LogitsWrapper(nn.Module):
    def __init__(self, model: MotorFault1DCNN):
        super().__init__()
        self.model = model
    def forward(self, signal: torch.Tensor) -> torch.Tensor:
        return self.model.encode_logits(signal)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)

# Fixed per-channel scale computed strictly on training set after DC removal
train_centered = X_train - np.mean(X_train, axis=-1, keepdims=True)
channel_scales = np.maximum(np.std(train_centered, axis=(0, 2), keepdims=True).astype(np.float32), 1e-8)

model = MotorFault1DCNN(channel_scales=channel_scales).to(device)
logits_net = LogitsWrapper(model).to(device)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=64,
    shuffle=True,
)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

best_val_acc = -1.0
best_state = None

for epoch in range(1, 31):
    logits_net.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(logits_net(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)

    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(y_val), 64):
            probs = model(torch.from_numpy(X_val[i:i+64]).to(device)).cpu().numpy()
            correct += int(np.sum(np.argmax(probs, axis=1) == y_val[i:i+64]))
    val_acc = correct / max(len(y_val), 1)

    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/30 | Loss: {total_loss/len(y_train):.4f} | Val Acc: {val_acc:.4f}")

model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
model_cpu = model.cpu().eval()

Training on: cuda
Epoch 01/30 | Loss: 1.7489 | Val Acc: 0.3558
Epoch 05/30 | Loss: 0.5522 | Val Acc: 0.5913
Epoch 10/30 | Loss: 0.2872 | Val Acc: 0.8429
Epoch 15/30 | Loss: 0.1814 | Val Acc: 0.8189
Epoch 20/30 | Loss: 0.1421 | Val Acc: 0.8606
Epoch 25/30 | Loss: 0.0694 | Val Acc: 0.9247
Epoch 30/30 | Loss: 0.0755 | Val Acc: 0.9311


In [ ]:
import shutil
import onnxruntime as ort
from sklearn.metrics import classification_report
from google.colab import files

# 1. Held-out test evaluation
preds_list = []
with torch.no_grad():
    for i in range(0, len(y_test), 64):
        probs = model_cpu(torch.from_numpy(X_test[i:i+64])).numpy()
        preds_list.append(np.argmax(probs, axis=1))
y_pred = np.concatenate(preds_list)

print("=== Held-Out Test Set Report (Grouped by Recording) ===")
print(classification_report(y_test, y_pred, target_names=CLASSES, digits=4, zero_division=0))

# 2. Export to local /content first (avoids Google Drive FUSE temp-file issues)
dummy_input = torch.randn(1, NUM_CHANNELS, WINDOW_SAMPLES, dtype=torch.float32)
torch.onnx.export(
    model_cpu,
    dummy_input,
    LOCAL_ONNX_PATH,
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=["signal"],
    output_names=["probabilities"],
    dynamic_axes=None,
)
print(f"Exported ONNX model locally to: {LOCAL_ONNX_PATH}")

# 3. Verify PyTorch vs ONNX Runtime parity
sess = ort.InferenceSession(LOCAL_ONNX_PATH, providers=["CPUExecutionProvider"])
max_err = 0.0
for i in range(min(8, len(X_test))):
    x_one = X_test[i : i + 1]
    with torch.no_grad():
        pt_out = model_cpu(torch.from_numpy(x_one)).numpy()
    ort_out = sess.run(["probabilities"], {"signal": x_one})[0]
    max_err = max(max_err, float(np.max(np.abs(pt_out - ort_out))))
print(f"Parity check passed | Max |PyTorch - ONNX| diff: {max_err:.2e}")

# 4. Copy to Drive & download
shutil.copy2(LOCAL_ONNX_PATH, DRIVE_ONNX_PATH)
print(f"Copied to Google Drive: {DRIVE_ONNX_PATH}")
files.download(LOCAL_ONNX_PATH)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# 1. Inspect current test confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix (rows = true, cols = pred):")
print(CLASSES)
print(cm)

Confusion Matrix (rows = true, cols = pred):
['H-H', 'R-U', 'R-M', 'B-R', 'K-A', 'S-W', 'V-U', 'F-B']
[[73  0  0  0  0  5  0  0]
 [ 0 64  0  0  0  0 14  0]
 [ 0  0 69  2  5  0  2  0]
 [ 0  0  5 34  0  0 39  0]
 [ 0  0  0  0 76  0  2  0]
 [ 0  0  4 12  0 56  6  0]
 [ 0  0  0  0  0  8 70  0]
 [ 0  0  3  0  0  0  1 74]]


In [ ]:
#code for v4 model Multi-scale 1D-CNN (kernel=15 + kernel=129 branches merged)

import shutil
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix
import onnxruntime as ort

# 1. Multi-Scale 1D-CNN (preserves exact [1, 4, 21000] -> [1, 8] contract)
class MultiScaleMotorFault1DCNN(nn.Module):
    def __init__(self, channel_scales: np.ndarray, num_classes: int = len(CLASSES)):
        super().__init__()
        scales = np.asarray(channel_scales, dtype=np.float32).reshape(1, NUM_CHANNELS, 1)
        inv_scale = 1.0 / np.maximum(scales, 1e-8)
        self.register_buffer("inv_scale", torch.tensor(inv_scale, dtype=torch.float32))

        # Branch A: fine temporal / high-frequency features (kernel=15)
        self.branch_fine = nn.Sequential(
            nn.Conv1d(NUM_CHANNELS, 24, kernel_size=15, stride=4, padding=7, bias=False),
            nn.BatchNorm1d(24),
            nn.ReLU(inplace=True),
            nn.AvgPool1d(kernel_size=4, stride=4),
        )

        # Branch B: broader waveform / modulation envelope (kernel=129)
        self.branch_broad = nn.Sequential(
            nn.Conv1d(NUM_CHANNELS, 24, kernel_size=129, stride=4, padding=64, bias=False),
            nn.BatchNorm1d(24),
            nn.ReLU(inplace=True),
            nn.AvgPool1d(kernel_size=4, stride=4),
        )

        # Merged multi-scale representation (48 channels)
        self.merged_block = nn.Sequential(
            nn.Conv1d(48, 64, kernel_size=16, stride=2, padding=8, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.AvgPool1d(kernel_size=4, stride=4),

            nn.Conv1d(64, 128, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool1d(1),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(64, num_classes),
        )
        self.softmax = nn.Softmax(dim=-1)

    def encode_logits(self, signal: torch.Tensor) -> torch.Tensor:
        centered = signal - signal.mean(dim=-1, keepdim=True)
        scaled = centered * self.inv_scale
        f_fine = self.branch_fine(scaled)
        f_broad = self.branch_broad(scaled)
        fused = torch.cat([f_fine, f_broad], dim=1)
        feats = self.merged_block(fused)
        return self.head(feats)

    def forward(self, signal: torch.Tensor) -> torch.Tensor:
        return self.softmax(self.encode_logits(signal))


class MultiScaleLogitsWrapper(nn.Module):
    def __init__(self, model: MultiScaleMotorFault1DCNN):
        super().__init__()
        self.model = model

    def forward(self, signal: torch.Tensor) -> torch.Tensor:
        return self.model.encode_logits(signal)


# 2. Collect all 128 recordings with recording group IDs
all_recordings = []
for root, _, files in os.walk(DATASET_DIR):
    for f in sorted(files):
        if f.lower().endswith(".mat") and not f.startswith("."):
            label = parse_fault_label(f)
            if label is not None:
                all_recordings.append((os.path.join(root, f), LABEL_TO_IDX[label]))

all_recordings.sort(key=lambda x: x[0])
rec_paths = [p for p, _ in all_recordings]
rec_labels = np.array([y for _, y in all_recordings], dtype=np.int64)
rec_groups = np.arange(len(all_recordings))  # Each recording file is its own group

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

best_overall_score = -1.0
best_overall_model = None
best_overall_fold = -1
best_val_split = None

for fold, (train_idx, val_idx) in enumerate(sgkf.split(rec_paths, rec_labels, rec_groups), start=1):
    tr_recs = [(rec_paths[i], int(rec_labels[i])) for i in train_idx]
    va_recs = [(rec_paths[i], int(rec_labels[i])) for i in val_idx]

    X_tr, y_tr = build_window_dataset(tr_recs)
    X_va, y_va = build_window_dataset(va_recs)

    tr_centered = X_tr - np.mean(X_tr, axis=-1, keepdims=True)
    ch_scales = np.maximum(np.std(tr_centered, axis=(0, 2), keepdims=True).astype(np.float32), 1e-8)

    fold_model = MultiScaleMotorFault1DCNN(channel_scales=ch_scales).to(device)
    fold_logits = MultiScaleLogitsWrapper(fold_model).to(device)
    optimizer = torch.optim.AdamW(fold_model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    loader = DataLoader(
        TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)),
        batch_size=64,
        shuffle=True,
    )

    best_fold_acc = -1.0
    best_fold_state = None

    for epoch in range(1, 31):
        fold_logits.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(fold_logits(xb), yb)
            loss.backward()
            optimizer.step()

        fold_model.eval()
        correct = 0
        with torch.no_grad():
            for i in range(0, len(y_va), 64):
                probs = fold_model(torch.from_numpy(X_va[i:i+64]).to(device)).cpu().numpy()
                correct += int(np.sum(np.argmax(probs, axis=1) == y_va[i:i+64]))
        va_acc = correct / max(len(y_va), 1)
        if va_acc >= best_fold_acc:
            best_fold_acc = va_acc
            best_fold_state = {k: v.cpu().clone() for k, v in fold_model.state_dict().items()}

    fold_model.load_state_dict({k: v.to(device) for k, v in best_fold_state.items()})
    fold_model_cpu = fold_model.cpu().eval()

    va_preds = []
    with torch.no_grad():
        for i in range(0, len(y_va), 64):
            va_preds.append(np.argmax(fold_model_cpu(torch.from_numpy(X_va[i:i+64])).numpy(), axis=1))
    va_preds = np.concatenate(va_preds)

    # Score folds using macro F1 across all 8 classes
    rep = classification_report(y_va, va_preds, target_names=CLASSES, output_dict=True, zero_division=0)
    macro_f1 = rep["macro avg"]["f1-score"]
    sw_f1 = rep["S-W"]["f1-score"]
    print(f"Fold {fold}/5 | Best Val Acc: {best_fold_acc:.4f} | Val Macro-F1: {macro_f1:.4f} | Val S-W F1: {sw_f1:.4f}")

    if macro_f1 > best_overall_score:
        best_overall_score = macro_f1
        best_overall_model = fold_model_cpu
        best_overall_fold = fold
        best_val_split = (X_va, y_va, va_preds)

print(f"\nSelected Best Model from Fold {best_overall_fold} (Val Macro-F1: {best_overall_score:.4f})")
X_va_best, y_va_best, va_preds_best = best_val_split
print("=== Best Fold Held-Out Validation Report ===")
print(classification_report(y_va_best, va_preds_best, target_names=CLASSES, digits=4, zero_division=0))
print("Confusion Matrix:")
print(confusion_matrix(y_va_best, va_preds_best))

# Export candidate v4 separately first so v1 stays untouched
V4_LOCAL_ONNX = "/content/motor_fault_classifier_v4.onnx"
V4_DRIVE_ONNX = "/content/drive/MyDrive/motor_fault_classifier_v4.onnx"

torch.onnx.export(
    best_overall_model,
    torch.randn(1, NUM_CHANNELS, WINDOW_SAMPLES, dtype=torch.float32),
    V4_LOCAL_ONNX,
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=["signal"],
    output_names=["probabilities"],
    dynamic_axes=None,
)

sess = ort.InferenceSession(V4_LOCAL_ONNX, providers=["CPUExecutionProvider"])
max_err = max(
    float(np.max(np.abs(
        best_overall_model(torch.from_numpy(X_va_best[i:i+1])).detach().numpy()
        - sess.run(["probabilities"], {"signal": X_va_best[i:i+1]})[0]
    )))
    for i in range(min(8, len(X_va_best)))
)
print(f"Exported {V4_LOCAL_ONNX} | Max parity diff: {max_err:.2e}")
shutil.copy2(V4_LOCAL_ONNX, V4_DRIVE_ONNX)
print(f"Copied to {V4_DRIVE_ONNX}")

Fold 1/5 | Best Val Acc: 0.8452 | Val Macro-F1: 0.8237 | Val S-W F1: 0.4400
Fold 2/5 | Best Val Acc: 0.9053 | Val Macro-F1: 0.8994 | Val S-W F1: 0.7919
Fold 3/5 | Best Val Acc: 0.7988 | Val Macro-F1: 0.8111 | Val S-W F1: 0.8278
Fold 4/5 | Best Val Acc: 0.8349 | Val Macro-F1: 0.8222 | Val S-W F1: 0.7146
Fold 5/5 | Best Val Acc: 0.8390 | Val Macro-F1: 0.8356 | Val S-W F1: 0.8889

Selected Best Model from Fold 2 (Val Macro-F1: 0.8994)
=== Best Fold Held-Out Validation Report ===
              precision    recall  f1-score   support

         H-H     0.9689    1.0000    0.9842       156
         R-U     0.9823    0.7115    0.8253       156
         R-M     0.6239    0.9359    0.7487        78
         B-R     0.9832    1.0000    0.9915       117
         K-A     0.9912    0.9658    0.9784       117
         S-W     0.9750    0.6667    0.7919       117
         V-U     0.7969    0.9808    0.8793       156
         F-B     0.9915    1.0000    0.9957       117

    accuracy                   

In [ ]:
#code for v4 best model Multi-scale 1D-CNN (kernel=15 + kernel=129 branches merged)

import shutil
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix
import onnxruntime as ort

# Ensure all 128 recordings are gathered consistently
all_recordings = []
for root, _, files in os.walk(DATASET_DIR):
    for f in sorted(files):
        if f.lower().endswith(".mat") and not f.startswith("."):
            label = parse_fault_label(f)
            if label is not None:
                all_recordings.append((os.path.join(root, f), LABEL_TO_IDX[label]))

all_recordings.sort(key=lambda x: x[0])
rec_paths = [p for p, _ in all_recordings]
rec_labels = np.array([y for _, y in all_recordings], dtype=np.int64)
rec_groups = np.arange(len(all_recordings))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(rec_paths, rec_labels, rec_groups), start=1):
    tr_recs = [(rec_paths[i], int(rec_labels[i])) for i in train_idx]
    va_recs = [(rec_paths[i], int(rec_labels[i])) for i in val_idx]

    X_tr, y_tr = build_window_dataset(tr_recs)
    X_va, y_va = build_window_dataset(va_recs)

    tr_centered = X_tr - np.mean(X_tr, axis=-1, keepdims=True)
    ch_scales = np.maximum(np.std(tr_centered, axis=(0, 2), keepdims=True).astype(np.float32), 1e-8)

    fold_model = MultiScaleMotorFault1DCNN(channel_scales=ch_scales).to(device)
    fold_logits = MultiScaleLogitsWrapper(fold_model).to(device)
    optimizer = torch.optim.AdamW(fold_model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    loader = DataLoader(
        TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)),
        batch_size=64,
        shuffle=True,
    )

    best_fold_acc = -1.0
    best_fold_state = None

    for epoch in range(1, 31):
        fold_logits.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(fold_logits(xb), yb)
            loss.backward()
            optimizer.step()

        fold_model.eval()
        correct = 0
        with torch.no_grad():
            for i in range(0, len(y_va), 64):
                probs = fold_model(torch.from_numpy(X_va[i:i+64]).to(device)).cpu().numpy()
                correct += int(np.sum(np.argmax(probs, axis=1) == y_va[i:i+64]))
        va_acc = correct / max(len(y_va), 1)
        if va_acc >= best_fold_acc:
            best_fold_acc = va_acc
            best_fold_state = {k: v.cpu().clone() for k, v in fold_model.state_dict().items()}

    fold_model.load_state_dict({k: v.to(device) for k, v in best_fold_state.items()})
    fold_model_cpu = fold_model.cpu().eval()

    va_preds = []
    with torch.no_grad():
        for i in range(0, len(y_va), 64):
            va_preds.append(np.argmax(fold_model_cpu(torch.from_numpy(X_va[i:i+64])).numpy(), axis=1))
    va_preds = np.concatenate(va_preds)

    rep = classification_report(y_va, va_preds, target_names=CLASSES, output_dict=True, zero_division=0)
    macro_f1 = rep["macro avg"]["f1-score"]
    sw_f1 = rep["S-W"]["f1-score"]
    print(f"Fold {fold}/5 | Val Acc: {best_fold_acc:.4f} | Macro-F1: {macro_f1:.4f} | S-W F1: {sw_f1:.4f}")

    fold_results.append({
        "fold": fold,
        "model": fold_model_cpu,
        "X_va": X_va,
        "y_va": y_va,
        "va_preds": va_preds,
        "val_acc": best_fold_acc,
        "macro_f1": macro_f1,
        "sw_f1": sw_f1,
    })

best_entry = max(fold_results, key=lambda r: 0.7 * r["macro_f1"] + 0.3 * r["sw_f1"])
print(
    f"\nSelected Fold {best_entry['fold']} | "
    f"Val Acc: {best_entry['val_acc']:.4f} | "
    f"Macro-F1: {best_entry['macro_f1']:.4f} | "
    f"S-W F1: {best_entry['sw_f1']:.4f}"
)
print(classification_report(
    best_entry["y_va"], best_entry["va_preds"], target_names=CLASSES, digits=4, zero_division=0
))
print("Confusion Matrix:")
print(confusion_matrix(best_entry["y_va"], best_entry["va_preds"]))

best_sw_model = best_entry["model"].cpu().eval()
V4_SW_ONNX = "/content/motor_fault_classifier_v4_best_sw.onnx"
V4_SW_DRIVE = "/content/drive/MyDrive/motor_fault_classifier_v4_best_sw_version_2.onnx"

torch.onnx.export(
    best_sw_model,
    torch.randn(1, NUM_CHANNELS, WINDOW_SAMPLES, dtype=torch.float32),
    V4_SW_ONNX,
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=["signal"],
    output_names=["probabilities"],
    dynamic_axes=None,
)

sess = ort.InferenceSession(V4_SW_ONNX, providers=["CPUExecutionProvider"])
X_va_best = best_entry["X_va"]
max_err = max(
    float(np.max(np.abs(
        best_sw_model(torch.from_numpy(X_va_best[i:i+1])).detach().numpy()
        - sess.run(["probabilities"], {"signal": X_va_best[i:i+1]})[0]
    )))
    for i in range(min(8, len(X_va_best)))
)
shutil.copy2(V4_SW_ONNX, V4_SW_DRIVE)
print(f"Exported {V4_SW_ONNX} & copied to {V4_SW_DRIVE} | Max parity diff: {max_err:.2e}")

Fold 1/5 | Val Acc: 0.8905 | Macro-F1: 0.8839 | S-W F1: 0.5982
Fold 2/5 | Val Acc: 0.9260 | Macro-F1: 0.9206 | S-W F1: 0.8000
Fold 3/5 | Val Acc: 0.8402 | Macro-F1: 0.8445 | S-W F1: 0.8125
Fold 4/5 | Val Acc: 0.9026 | Macro-F1: 0.8946 | S-W F1: 0.7784
Fold 5/5 | Val Acc: 0.8318 | Macro-F1: 0.8332 | S-W F1: 0.9182

Selected Fold 2 | Val Acc: 0.9260 | Macro-F1: 0.9206 | S-W F1: 0.8000
              precision    recall  f1-score   support

         H-H     0.9341    1.0000    0.9659       156
         R-U     0.9929    0.8974    0.9428       156
         R-M     0.8837    0.9744    0.9268        78
         B-R     0.7971    0.9402    0.8627       117
         K-A     0.8417    1.0000    0.9141       117
         S-W     1.0000    0.6667    0.8000       117
         V-U     0.9864    0.9295    0.9571       156
         F-B     0.9915    1.0000    0.9957       117

    accuracy                         0.9260      1014
   macro avg     0.9284    0.9260    0.9206      1014
weighted avg     0

In [ ]:
#code for v5 model 3-fold probability average (Inner Folds 2, 4, 3 of multi-scale 1D-CNN)

import shutil
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
import onnxruntime as ort

class MultiScaleEnsemble3(nn.Module):
    def __init__(self, models_list):
        super().__init__()
        self.fold_models = nn.ModuleList([m.cpu().eval() for m in models_list])

    def forward(self, signal: torch.Tensor) -> torch.Tensor:
        probs_sum = None
        for m in self.fold_models:
            p = m(signal)
            probs_sum = p if probs_sum is None else (probs_sum + p)
        return probs_sum / float(len(self.fold_models))

# Pick Folds 2, 4, and 5 from fold_results
target_folds = {2, 4, 5}
selected_entries = [r for r in fold_results if r["fold"] in target_folds]
assert len(selected_entries) == 3, f"Expected 3 folds, got {[r['fold'] for r in selected_entries]}"

ens_model = MultiScaleEnsemble3([r["model"] for r in selected_entries]).cpu().eval()

# Evaluate the 3-fold ensemble across each of those 3 held-out validation splits
all_va_X, all_va_y, all_va_preds = [], [], []
for r in selected_entries:
    X_va_f = r["X_va"]
    y_va_f = r["y_va"]
    preds_f = []
    with torch.no_grad():
        for i in range(0, len(y_va_f), 64):
            p = ens_model(torch.from_numpy(X_va_f[i:i+64])).numpy()
            preds_f.append(np.argmax(p, axis=1))
    preds_f = np.concatenate(preds_f)
    rep_f = classification_report(y_va_f, preds_f, target_names=CLASSES, output_dict=True, zero_division=0)
    print(
        f"Ensemble on Fold {r['fold']} Val ({len(y_va_f)} win) | "
        f"Acc: {rep_f['accuracy']:.4f} | Macro-F1: {rep_f['macro avg']['f1-score']:.4f} | "
        f"S-W F1: {rep_f['S-W']['f1-score']:.4f}"
    )
    all_va_X.append(X_va_f)
    all_va_y.append(y_va_f)
    all_va_preds.append(preds_f)

comb_X = np.concatenate(all_va_X, axis=0)
comb_y = np.concatenate(all_va_y, axis=0)
comb_preds = np.concatenate(all_va_preds, axis=0)

print("\n=== 3-Fold Ensemble Combined Held-Out Validation Report (Folds 2, 4, 5) ===")
print(classification_report(comb_y, comb_preds, target_names=CLASSES, digits=4, zero_division=0))
print("Confusion Matrix:")
print(confusion_matrix(comb_y, comb_preds))

# Export single combined ONNX model
V5_LOCAL_ONNX = "/content/motor_fault_classifier_v5_ens3.onnx"
V5_DRIVE_ONNX = "/content/drive/MyDrive/motor_fault_classifier_v5_ens3.onnx"

torch.onnx.export(
    ens_model,
    torch.randn(1, NUM_CHANNELS, WINDOW_SAMPLES, dtype=torch.float32),
    V5_LOCAL_ONNX,
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=["signal"],
    output_names=["probabilities"],
    dynamic_axes=None,
)

sess = ort.InferenceSession(V5_LOCAL_ONNX, providers=["CPUExecutionProvider"])
max_err = max(
    float(np.max(np.abs(
        ens_model(torch.from_numpy(comb_X[i:i+1])).detach().numpy()
        - sess.run(["probabilities"], {"signal": comb_X[i:i+1]})[0]
    )))
    for i in range(min(8, len(comb_X)))
)
shutil.copy2(V5_LOCAL_ONNX, V5_DRIVE_ONNX)
print(f"\nExported {V5_LOCAL_ONNX} & copied to {V5_DRIVE_ONNX} | Max parity diff: {max_err:.2e}")

Ensemble on Fold 2 Val (1014 win) | Acc: 0.9951 | Macro-F1: 0.9952 | S-W F1: 1.0000
Ensemble on Fold 4 Val (975 win) | Acc: 0.9897 | Macro-F1: 0.9874 | S-W F1: 0.9870
Ensemble on Fold 5 Val (975 win) | Acc: 0.9938 | Macro-F1: 0.9932 | S-W F1: 0.9804

=== 3-Fold Ensemble Combined Held-Out Validation Report (Folds 2, 4, 5) ===
              precision    recall  f1-score   support

         H-H     1.0000    1.0000    1.0000       429
         R-U     0.9972    1.0000    0.9986       351
         R-M     1.0000    0.9949    0.9974       390
         B-R     1.0000    0.9972    0.9986       351
         K-A     0.9693    0.9886    0.9788       351
         S-W     1.0000    0.9801    0.9899       351
         V-U     0.9770    0.9821    0.9795       390
         F-B     1.0000    1.0000    1.0000       351

    accuracy                         0.9929      2964
   macro avg     0.9929    0.9928    0.9929      2964
weighted avg     0.9930    0.9929    0.9929      2964

Confusion Matrix:
[[42

In [ ]:
#check overlapping
import numpy as np
import torch
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix

# Reconstruct exact fold indices from the same seed=42 split
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
val_sets_by_fold = {}
for fold, (_, val_idx) in enumerate(sgkf.split(rec_paths, rec_labels, rec_groups), start=1):
    val_sets_by_fold[fold] = set(val_idx.tolist())

unseen_by_all_3 = sorted(val_sets_by_fold[2] & val_sets_by_fold[4] & val_sets_by_fold[5])
print(f"Recordings unseen by ALL of Folds 2, 4, and 5: {len(unseen_by_all_3)}")

if len(unseen_by_all_3) > 0:
    strict_recs = [(rec_paths[i], int(rec_labels[i])) for i in unseen_by_all_3]
    X_strict, y_strict = build_window_dataset(strict_recs)
    ens_model.eval()
    preds_strict = []
    with torch.no_grad():
        for i in range(0, len(y_strict), 64):
            p = ens_model(torch.from_numpy(X_strict[i:i+64])).numpy()
            preds_strict.append(np.argmax(p, axis=1))
    preds_strict = np.concatenate(preds_strict)
    print(classification_report(y_strict, preds_strict, target_names=CLASSES, digits=4, zero_division=0))
    print(confusion_matrix(y_strict, preds_strict))

Recordings unseen by ALL of Folds 2, 4, and 5: 0


In [ ]:
#code for v5 model locked version 3-fold probability average (Inner Folds 2, 4, 3 of multi-scale 1D-CNN)
import shutil
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
import onnxruntime as ort

# 1. Lock aside 16 held-out test recordings (2 per class)
gss = GroupShuffleSplit(n_splits=1, test_size=16, random_state=101)
dev_idx, test_idx = next(gss.split(rec_paths, rec_labels, rec_groups))

# Verify class balance of held-out test set; fallback to 2-per-class stratified pick if needed
test_by_cls = {c: [] for c in range(len(CLASSES))}
for idx in test_idx:
    test_by_cls[int(rec_labels[idx])].append(idx)

if all(len(v) == 2 for v in test_by_cls.values()):
    locked_test_idx = np.array(sorted([i for v in test_by_cls.values() for i in v]))
    locked_dev_idx = np.array([i for i in range(len(rec_paths)) if i not in set(locked_test_idx)])
else:
    rng = np.random.default_rng(101)
    picked = []
    for c in range(len(CLASSES)):
        c_indices = [i for i in range(len(rec_paths)) if rec_labels[i] == c]
        picked.extend(rng.choice(c_indices, size=2, replace=False).tolist())
    locked_test_idx = np.array(sorted(picked))
    locked_dev_idx = np.array([i for i in range(len(rec_paths)) if i not in set(locked_test_idx)])

dev_paths = [rec_paths[i] for i in locked_dev_idx]
dev_labels = rec_labels[locked_dev_idx]
dev_groups = np.arange(len(dev_paths))

test_recs = [(rec_paths[i], int(rec_labels[i])) for i in locked_test_idx]
X_test_locked, y_test_locked = build_window_dataset(test_recs)
print(f"Locked Dev recordings: {len(dev_paths)} | Locked Held-Out Test recordings: {len(test_recs)} ({len(y_test_locked)} windows)")

# 2. 5-fold recording-grouped CV strictly on the 112 dev recordings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
inner_folds = []

for fold, (tr_i, va_i) in enumerate(inner_cv.split(dev_paths, dev_labels, dev_groups), start=1):
    tr_recs = [(dev_paths[i], int(dev_labels[i])) for i in tr_i]
    va_recs = [(dev_paths[i], int(dev_labels[i])) for i in va_i]

    X_tr, y_tr = build_window_dataset(tr_recs)
    X_va, y_va = build_window_dataset(va_recs)

    tr_centered = X_tr - np.mean(X_tr, axis=-1, keepdims=True)
    ch_scales = np.maximum(np.std(tr_centered, axis=(0, 2), keepdims=True).astype(np.float32), 1e-8)

    m = MultiScaleMotorFault1DCNN(channel_scales=ch_scales).to(device)
    lw = MultiScaleLogitsWrapper(m).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()

    loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)), batch_size=64, shuffle=True)
    best_acc, best_st = -1.0, None

    for epoch in range(1, 31):
        lw.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(lw(xb), yb)
            loss.backward()
            opt.step()

        m.eval()
        corr = 0
        with torch.no_grad():
            for i in range(0, len(y_va), 64):
                pr = m(torch.from_numpy(X_va[i:i+64]).to(device)).cpu().numpy()
                corr += int(np.sum(np.argmax(pr, axis=1) == y_va[i:i+64]))
        va_acc = corr / max(len(y_va), 1)
        if va_acc >= best_acc:
            best_acc = va_acc
            best_st = {k: v.cpu().clone() for k, v in m.state_dict().items()}

    m.load_state_dict({k: v.to(device) for k, v in best_st.items()})
    m_cpu = m.cpu().eval()

    va_preds = []
    with torch.no_grad():
        for i in range(0, len(y_va), 64):
            va_preds.append(np.argmax(m_cpu(torch.from_numpy(X_va[i:i+64])).numpy(), axis=1))
    va_preds = np.concatenate(va_preds)

    rep = classification_report(y_va, va_preds, target_names=CLASSES, output_dict=True, zero_division=0)
    macro_f1 = rep["macro avg"]["f1-score"]
    sw_f1 = rep["S-W"]["f1-score"]
    print(f"Inner Fold {fold}/5 | Val Acc: {best_acc:.4f} | Macro-F1: {macro_f1:.4f} | S-W F1: {sw_f1:.4f}")

    inner_folds.append({
        "fold": fold,
        "model": m_cpu,
        "score": 0.7 * macro_f1 + 0.3 * sw_f1,
    })

# 3. Top-3 fold ensemble evaluated strictly on the locked 16-recording test set
top3 = sorted(inner_folds, key=lambda d: d["score"], reverse=True)[:3]
print("Ensembling top-3 inner folds:", [d["fold"] for d in top3])

locked_ens = MultiScaleEnsemble3([d["model"] for d in top3]).cpu().eval()
test_preds = []
with torch.no_grad():
    for i in range(0, len(y_test_locked), 64):
        p = locked_ens(torch.from_numpy(X_test_locked[i:i+64])).numpy()
        test_preds.append(np.argmax(p, axis=1))
test_preds = np.concatenate(test_preds)

print("\n=== STRICT LOCKED HELD-OUT TEST REPORT (16 Recordings / 624 Windows) ===")
print(classification_report(y_test_locked, test_preds, target_names=CLASSES, digits=4, zero_division=0))
print("Confusion Matrix:")
print(confusion_matrix(y_test_locked, test_preds))

# 4. Export & parity check
LOCKED_LOCAL_ONNX = "/content/motor_fault_classifier_v5_locked_test.onnx"
LOCKED_DRIVE_ONNX = "/content/drive/MyDrive/motor_fault_classifier_v5_locked_test.onnx"

torch.onnx.export(
    locked_ens,
    torch.randn(1, NUM_CHANNELS, WINDOW_SAMPLES, dtype=torch.float32),
    LOCKED_LOCAL_ONNX,
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=["signal"],
    output_names=["probabilities"],
    dynamic_axes=None,
)

sess = ort.InferenceSession(LOCKED_LOCAL_ONNX, providers=["CPUExecutionProvider"])
max_err = max(
    float(np.max(np.abs(
        locked_ens(torch.from_numpy(X_test_locked[i:i+1])).detach().numpy()
        - sess.run(["probabilities"], {"signal": X_test_locked[i:i+1]})[0]
    )))
    for i in range(min(8, len(X_test_locked)))
)
shutil.copy2(LOCKED_LOCAL_ONNX, LOCKED_DRIVE_ONNX)
print(f"Exported {LOCKED_LOCAL_ONNX} & copied to {LOCKED_DRIVE_ONNX} | Max parity diff: {max_err:.2e}")

Locked Dev recordings: 112 | Locked Held-Out Test recordings: 16 (624 windows)
Inner Fold 1/5 | Val Acc: 0.6867 | Macro-F1: 0.7006 | S-W F1: 0.2160
Inner Fold 2/5 | Val Acc: 0.9264 | Macro-F1: 0.9240 | S-W F1: 0.9494
Inner Fold 3/5 | Val Acc: 0.8753 | Macro-F1: 0.8724 | S-W F1: 0.6417
Inner Fold 4/5 | Val Acc: 0.8495 | Macro-F1: 0.8372 | S-W F1: 0.7868
Inner Fold 5/5 | Val Acc: 0.8120 | Macro-F1: 0.7405 | S-W F1: 0.8659
Ensembling top-3 inner folds: [2, 4, 3]

=== STRICT LOCKED HELD-OUT TEST REPORT (16 Recordings / 624 Windows) ===
              precision    recall  f1-score   support

         H-H     1.0000    1.0000    1.0000        78
         R-U     1.0000    0.9359    0.9669        78
         R-M     0.9873    1.0000    0.9936        78
         B-R     0.7959    1.0000    0.8864        78
         K-A     1.0000    0.9615    0.9804        78
         S-W     0.8750    0.8077    0.8400        78
         V-U     0.9718    0.8846    0.9262        78
         F-B     1.0000    1.